In [1]:
import sys
import os
import json
from datetime import datetime, timezone
from pathlib import Path

# Adiciona a raiz do projeto ao PATH do Python para importar utils
sys.path.append(str(Path.cwd().parent))

from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import execute_values

from utils.file_utils import FileUtils

In [2]:
# Correção do __file__ e do erro de digitação "open_mete"
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

SOURCE_DIR = PROJECT_ROOT / "data" / "bronze" / "open_meteo" / "forecast"

print("Source directory:", SOURCE_DIR)
print("UTC now:", datetime.now(timezone.utc).isoformat())

Source directory: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast
UTC now: 2026-08-03T21:56:37.284593+00:00


In [3]:
def connect():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST"),
        port=os.getenv("POSTGRES_PORT"),
        dbname=os.getenv("POSTGRES_DATABASE"),
        user=os.getenv("POSTGRES_USERNAME"),
        password=os.getenv("POSTGRES_PASSWORD"),
    )

def get_forecast_files(directory: Path):
    return sorted(directory.rglob("*.json"))

def build_forecast_rows(path: Path) -> list[tuple]:
    raw = FileUtils.read_json(path)
    hourly = raw.get("hourly", {})
    times = hourly.get("time", [])
    if not isinstance(times, list):
        return []

    fields = [
        "temperature_2m",
        "apparent_temperature",
        "relative_humidity_2m",
        "precipitation_probability",
        "precipitation",
        "weather_code",
        "wind_speed_10m",
        "wind_direction_10m",
    ]

    columns = [hourly.get(field, []) for field in fields]
    if any(not isinstance(column, list) for column in columns):
        return []
    if any(len(column) != len(times) for column in columns):
        return []

    city_name = path.stem.rsplit("_", 1)[0]
    ingestion_date = path.parent.name
    rows = []
    for idx, observation_time in enumerate(times):
        rows.append((
            "open_meteo", # source_name
            "forecast",   # source_type
            city_name,
            ingestion_date,
            f"{observation_time}Z",
            columns[0][idx],
            columns[1][idx],
            columns[2][idx],
            columns[3][idx],
            columns[4][idx],
            columns[5][idx],
            columns[6][idx],
            columns[7][idx],
        ))
    return rows

In [4]:
# Correção dos nomes das variáveis (paths -> forecast_files, build_rows -> build_forecast_rows)
forecast_files = get_forecast_files(SOURCE_DIR)

all_rows = []
skipped = []

for path in forecast_files:
    rows = build_forecast_rows(path)
    if rows:
        all_rows.extend(rows)
    else:
        skipped.append(str(path))

print(f"Found {len(forecast_files)} forecast files")
print(f"Built {len(all_rows)} rows")
if skipped:
    print(f"Skipped {len(skipped)} invalid files")
    for path in skipped[:10]:
        print(path)

Found 60 forecast files
Built 10080 rows


In [ ]:
insert_sql = """
IINSERT INTO silver.forecast_weather_hourly (
    source_name, 
    source_type, 
    city_name, 
    ingestion_date, 
    observation_time,
    temperature_celsius, 
    feels_like_celsius, 
    relative_humidity_pct,
    precipitation_probability, 
    precipitation_mm, 
    weather_code,
    wind_speed_10m, 
    wind_direction_10m
) VALUES %s;
"""

if all_rows:
    with connect() as conn:
        with conn.cursor() as cursor:
            execute_values(cursor, insert_sql, all_rows)
        conn.commit()
    print("Loaded rows into silver.forecast_weather_hourly")
else:
    print("Nenhuma linha para inserir.")

Loaded rows into silver.forecast_weather_hourly
